# Name That LLM: TF-IDF + stylometry LinearSVC with a style-matched Open Pool sample + 4 two-stage LoRA classifiers

Generated by `name_that_llm/kaggle_notebook.py`; do not edit by hand.

* **Model:** character 1-5-grams + word 1-2-grams + 16 punctuation/style rates, then LinearSVC.
* **Data:** `train.csv` plus 2,000 passages per model from the public [Open Pool](https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool), sampled by its `match_weight` so typography and length match the real passages.
* **LoRA members:** checkpoints from the training notebooks, blended by adding log-probabilities with weights `lora_out_4b_full` 0.25, `lora_out_8b_full` 0.5, `lora_out_mistral7b_full` 0.5, `lora_out_14b_full` 1.0.
* **GLM-5.3 likelihood member:** the 20% of test passages the blend most suspects of being GLM, scored under the real GLM-5.3 and GLM-5.3-Flash; weight 2.0.
* **Post-processing:** partial Sinkhorn class balancing over the test set.
* **Local grouped CV** (5 folds by `prompt_id`): balanced accuracy 0.451 for the SVM alone.

Needs GPU T4 x2 and no internet; the GLM stage takes about 25 minutes plus 5 s per passage scored, as many as fit before 8.5 h.

In [ ]:
import gc
import glob
import json
import os
import re
import time

T_START = time.time()  # the GLM stage stops in time for the 9 h limit
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    # find the inputs wherever Kaggle mounts them; list /kaggle/input if one is missing
    attached = sorted(glob.glob('/kaggle/input/*/*', recursive=False))
    print('attached inputs:', attached)
    comps = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), 'test.csv'))]
    assert comps, f'Attach the name-that-llm competition data; found {attached}'
    COMP = comps[0]
    pools = glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True)
    assert pools, f'Attach crtproduction/name-that-llm-open-pool; found {attached}'
    POOL = pools[0]
    OUT = 'submission.parquet'
else:  # running from this project's notebooks/ folder
    COMP = '../data/raw'
    POOL = '../data/external/open_pool/ntl_open_pool.parquet'
    OUT = '../data/processed/submission_from_notebook.parquet'

# settings of the committed local model (reports/pool2000_w1_cv_metrics.json)
N_PER_LABEL, POOL_SEED, POOL_WEIGHT = 2000, 0, 1.0
C, STYLE_WEIGHT = 0.5, 0.2
BALANCE_TEMPERATURE, BALANCE_STRENGTH = 1.0, 0.3
USE_LORA, LORA_WEIGHTS, T_SVM = True, {'lora_out_4b_full': 0.25, 'lora_out_8b_full': 0.5, 'lora_out_mistral7b_full': 0.5, 'lora_out_14b_full': 1.0}, 2.0
print('competition data:', COMP)
print('open pool:', POOL)
# LoRA members: each training notebook's output folder (lora_out*) + its base model.
# Checkpoints saved before meta.json recorded the base model are Qwen3-4B-Base.
DEFAULT_BASE = {'model_type': 'qwen3', 'hidden_size': 2560, 'num_hidden_layers': 36}


def find_base(spec):
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f'attach the base model {spec}')


if ON_KAGGLE:
    # a member uploaded as a dataset sits in a folder named after the dataset, with dashes
    found = {os.path.basename(os.path.dirname(p)).replace('-', '_'): os.path.dirname(p)
             for p in glob.glob('/kaggle/input/**/lora_classifier.safetensors', recursive=True)}
else:  # NTL_LORA_DIRS='name=path,...'; one base model (NTL_LORA_BASE) for all of them
    found = dict(kv.split('=', 1) for kv in os.environ.get(
        'NTL_LORA_DIRS', 'lora_out=../data/interim/lora4b').split(','))
missing = sorted(set(LORA_WEIGHTS) - set(found))
assert not missing, f'Attach the LoRA training notebook outputs {missing}; found {found}'
LORA_DIRS = {name: found[name] for name in LORA_WEIGHTS}
# a 5-fold member keeps its fold adapters in fold0..fold4 next to its full fit; average them
LORA_ADAPTERS = {name: [d] + sorted(glob.glob(os.path.join(d, 'fold[0-9]')))
                 for name, d in LORA_DIRS.items()}
for name, d in LORA_DIRS.items():
    print(f'LoRA member {name} (weight {LORA_WEIGHTS[name]}): {d}, '
          f'{len(LORA_ADAPTERS[name])} adapter(s)')


## Features (copied from `name_that_llm/features.py`)

In [ ]:
import re

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import StandardScaler

# name -> regex; counted per 100 words
PUNCT_PATTERNS = {
    "em_dash": "—",
    "en_dash": "–",
    "semicolon": ";",
    "colon": ":",
    "comma": ",",
    "question": r"\?",
    "exclaim": "!",
    "parens": r"\(",
    "curly_quote": "[‘’“”]",
    "straight_quote": "['\"]",
    "digits": r"\d",
}
_SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")
_WORD = re.compile(r"[a-z']+")


def style_features(texts) -> pd.DataFrame:
    """Dense stylometric features: punctuation rates, length and lexical diversity."""
    s = pd.Series(list(texts), dtype=object)
    words = s.str.split().str.len().clip(lower=1)
    out = {name: s.str.count(pat) / words * 100 for name, pat in PUNCT_PATTERNS.items()}
    out["has_em_dash"] = s.str.contains("—").astype(float)
    out["has_curly"] = s.str.contains("[‘’“”]").astype(float)
    out["log_words"] = np.log(words)
    sentences = s.map(lambda t: max(1, len([x for x in _SENTENCE_SPLIT.split(t.strip()) if x])))
    out["words_per_sentence"] = words / sentences
    tokens = s.str.lower().map(_WORD.findall)
    out["avg_word_len"] = tokens.map(lambda w: np.mean([len(x) for x in w]) if w else 0.0)
    out["type_token"] = tokens.map(lambda w: len(set(w)) / max(1, len(w)))
    return pd.DataFrame(out).fillna(0.0)


class StyleFeatures(BaseEstimator, TransformerMixin):
    """sklearn wrapper around :func:`style_features`."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return style_features(X).to_numpy(dtype=float)

    def get_feature_names_out(self, input_features=None):
        return np.array(style_features(["a b."]).columns, dtype=object)


def build_features(style_weight: float = 1.0) -> FeatureUnion:
    """Character n-grams + word n-grams + scaled stylometric features."""
    return FeatureUnion(
        [
            (
                "char",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(1, 5),
                    lowercase=False,
                    sublinear_tf=True,
                    min_df=2,
                    max_features=300_000,
                    dtype=np.float32,
                ),
            ),
            (
                "word",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    lowercase=True,
                    token_pattern=r"(?u)\b\w[\w']*\b",
                    sublinear_tf=True,
                    min_df=2,
                    dtype=np.float32,
                ),
            ),
            ("style", make_pipeline(StyleFeatures(), StandardScaler())),
        ],
        transformer_weights={"char": 1.0, "word": 1.0, "style": style_weight},
    )


## Class balancing (copied from `name_that_llm/modeling/postprocess.py`)

In [ ]:
def softmax(scores: np.ndarray, temperature: float) -> np.ndarray:
    z = scores * temperature
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def sinkhorn(proba: np.ndarray, strength: float = 1.0, n_iter: int = 50) -> np.ndarray:
    """Geometric mix of ``proba`` and its column-balanced Sinkhorn projection."""
    if strength <= 0:
        return proba
    n, k = proba.shape
    q = proba + 1e-12
    for _ in range(n_iter):
        q = q / q.sum(axis=0, keepdims=True) * (n / k)  # each class gets n/k total mass
        q = q / q.sum(axis=1, keepdims=True)  # each row is a distribution
    logp = (1 - strength) * np.log(proba + 1e-12) + strength * np.log(q)
    p = np.exp(logp - logp.max(axis=1, keepdims=True))
    return p / p.sum(axis=1, keepdims=True)


## Data: train.csv + a style-matched Open Pool sample

In [ ]:
train = pd.read_csv(f'{COMP}/train.csv')
test = pd.read_csv(f'{COMP}/test.csv')
pool = pd.read_parquet(POOL, columns=['text', 'model', 'match_weight'])

# same draw as name_that_llm/dataset.py: weighted sampling without replacement
# (Efraimidis-Spirakis) by match_weight, the same number of passages per label
parts = []
for label, d in pool.groupby('model'):
    rng = np.random.default_rng([POOL_SEED, len(parts)])
    keys = rng.random(len(d)) ** (1.0 / d.match_weight.clip(lower=1e-6).to_numpy())
    parts.append(d.iloc[np.argsort(-keys)[:min(N_PER_LABEL, len(d))]])
pool_sample = pd.concat(parts, ignore_index=True)
del pool

LABELS = sorted(train.model.unique())
assert len(LABELS) == 18 and set(pool_sample.model) <= set(LABELS)
print(f'train {len(train)}, pool sample {len(pool_sample)}, test {len(test)}')

## Train

In [ ]:
X = pd.concat([train.text, pool_sample.text], ignore_index=True)
y = np.concatenate([train.model.to_numpy(), pool_sample.model.to_numpy()])
w = np.concatenate([np.ones(len(train)), np.full(len(pool_sample), POOL_WEIGHT)])

model = Pipeline([
    ('features', build_features(STYLE_WEIGHT)),
    ('clf', LinearSVC(C=C, class_weight='balanced', max_iter=5000)),
])
t0 = time.time()
model.fit(X, y, clf__sample_weight=w)
print(f'trained in {time.time() - t0:.0f}s; in-sample balanced acc on train.csv: '
      f'{np.mean([np.mean(model.predict(train.text[train.model == m]) == m) for m in LABELS]):.3f}')

## SVM scores for the test set

In [ ]:
svm_scores = model.decision_function(test.text)
CLASSES = np.asarray(model.classes_)
del model, X, y, w, pool_sample
gc.collect()

## LoRA classifier (copied from `name_that_llm/lora.py`)

In [ ]:
import contextlib
import json
import math
import os
import time

import numpy as np
import torch
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update ``B @ A * alpha / r``."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        dev = base.weight.device
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features, device=dev))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r, device=dev))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        delta = self.drop(x).to(self.lora_A.dtype) @ self.lora_A.T @ self.lora_B.T
        return self.base(x) + (delta * self.scale).to(x.dtype)


def add_lora(model: nn.Module, r: int, alpha: float, dropout: float) -> int:
    """Wrap every target linear layer of ``model`` in LoRALinear; returns how many."""
    targets = [
        (parent, name)
        for parent in model.modules()
        for name, child in parent.named_children()
        if isinstance(child, nn.Linear) and name in LORA_TARGETS
    ]
    for parent, name in targets:
        setattr(parent, name, LoRALinear(getattr(parent, name), r, alpha, dropout))
    return len(targets)


def _to_device(x, device):
    """``x`` with every tensor in it (also inside tuples, lists and dicts) on ``device``."""
    if torch.is_tensor(x):
        return x.to(device)
    if isinstance(x, (tuple, list)):
        return type(x)(_to_device(v, device) for v in x)
    if isinstance(x, dict):
        return {k: _to_device(v, device) for k, v in x.items()}
    return x


def split_trunk(trunk: nn.Module, device, device2) -> int:
    """Put the embeddings and the first decoder layers on ``device`` and the other layers on
    ``device2``, about half of the weights on each; returns how many layers stay on ``device``.
    Each moved layer first brings its inputs (hidden states, mask, rotary tables) to its device."""
    size = lambda m: sum(q.numel() * q.element_size() for q in m.parameters())
    half, used, first = size(trunk) / 2, size(trunk.embed_tokens), 0
    for layer in trunk.layers:
        if used + size(layer) > half and first > 0:
            break
        used, first = used + size(layer), first + 1
    trunk.embed_tokens.to(device)
    for i, layer in enumerate(trunk.layers):
        layer.to(device if i < first else device2)
        if i >= first:
            layer.register_forward_pre_hook(
                lambda mod, args, kwargs: (_to_device(args, device2), _to_device(kwargs, device2)),
                with_kwargs=True,
            )
    return first


def load_trunk(
    base_dir: str, keep: int, dtype, device, attn_implementation=None, device2=None
) -> nn.Module:
    """The first ``keep`` decoder layers of the base model, frozen, final norm switched off.

    With ``device2`` (inference only), a trunk too large for ``device`` is split over the two
    (``split_trunk``): the 24-layer Qwen3-14B trunk is ~19 GB in fp16, a T4 holds 15 GB."""
    cfg = AutoConfig.from_pretrained(base_dir)
    cfg.num_hidden_layers, cfg.use_cache = keep, False
    if getattr(cfg, "layer_types", None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    kw = {"attn_implementation": attn_implementation} if attn_implementation else {}
    trunk = AutoModel.from_pretrained(base_dir, config=cfg, dtype=dtype, **kw)
    # .to(dtype) too: transformers before 4.56 ignores ``dtype=`` and would load fp32
    trunk = trunk.to(dtype=dtype)
    trunk.norm = nn.Identity()
    trunk.requires_grad_(False)
    nbytes = sum(q.numel() * q.element_size() for q in trunk.parameters())
    room = None
    if device2 is not None and torch.device(device).type == "cuda":
        room = torch.cuda.get_device_properties(device).total_memory
    if device2 is not None and (room is None or nbytes > 0.8 * room):
        first = split_trunk(trunk, device, device2)
        print(f"trunk of {nbytes / 2**30:.1f} GiB split: embeddings and layers 0-{first - 1} "
              f"on {device}, layers {first}-{keep - 1} on {device2}")  # fmt: skip
        return trunk
    return trunk.to(device)


def no_autocast(device_type: str):
    """Autocast switched off (the head runs in fp32); XLA runs without autocast anyway."""
    if device_type in ("cuda", "cpu"):
        return torch.autocast(device_type=device_type, enabled=False)
    return contextlib.nullcontext()


class LoRAClassifier(nn.Module):
    def __init__(
        self, trunk: nn.Module, layers, n_classes: int, r=16, alpha=32, dropout=0.05, ckpt=True
    ):
        super().__init__()
        self.trunk, self.layers = trunk, list(layers)
        self.n_lora = add_lora(trunk, r, alpha, dropout)
        if ckpt:  # recompute activations in backward: far less memory for ~30% more time
            trunk.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )
        H = trunk.config.hidden_size
        dev = trunk.embed_tokens.weight.device
        self.norms = nn.ModuleList([nn.LayerNorm(H) for _ in self.layers]).to(dev)
        self.head = nn.Linear(H * len(self.layers), n_classes).to(dev)

    def forward(self, input_ids, attention_mask):
        hs = self.trunk(
            input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True
        ).hidden_states
        m = attention_mask.unsqueeze(-1).float()
        m[:, 0] = 0  # the first token is an attention sink; leave it out of the mean
        den = m.sum(1).clamp(min=1)
        with no_autocast(input_ids.device.type):
            z = torch.cat(
                # .to: a split trunk returns its later hidden states on the second device
                [
                    nm((hs[l].to(m.device).float() * m).sum(1) / den)
                    for l, nm in zip(self.layers, self.norms)
                ],
                -1,
            )
            return self.head(z)

    def trainable_state(self) -> dict:
        """Only what training changes: LoRA matrices, LayerNorms and the head."""
        return {k: v.detach().cpu() for k, v in self.named_parameters() if v.requires_grad}


def tokenize(tok, texts, max_len: int) -> list[list[int]]:
    """Token ids of each passage, ending in EOS, and starting with BOS if the tokenizer has one
    (Llama does; Qwen has none, so Qwen ids are just the text and EOS). At most ``max_len``."""
    bos = [] if tok.bos_token_id is None else [tok.bos_token_id]
    ids = tok(
        list(texts),
        add_special_tokens=False,
        truncation=True,
        max_length=max_len - 1 - len(bos),
    )
    return [bos + x + [tok.eos_token_id] for x in ids["input_ids"]]


def batches(ids, bs: int, shuffle: bool, rng=None):
    """Index batches of similar length (less padding); batch order shuffled when training."""
    if not shuffle:
        order = np.argsort([len(x) for x in ids], kind="stable")
    else:  # sort within random chunks of 50 batches so batches still differ between epochs
        order = rng.permutation(len(ids))
        chunk = bs * 50
        order = np.concatenate(
            [
                sorted(order[i : i + chunk], key=lambda j: len(ids[j]))
                for i in range(0, len(order), chunk)
            ]
        )
    out = [order[i : i + bs] for i in range(0, len(order), bs)]
    if shuffle:
        out = [out[i] for i in rng.permutation(len(out))]
    return out


PAD_MULTIPLE = 64  # static shapes: sequence lengths are rounded up to a multiple of this


def static_len(ids, idx, max_len: int) -> int:
    """The padded length for a batch: its longest passage rounded up to ``PAD_MULTIPLE``, so XLA
    sees at most ``max_len / PAD_MULTIPLE`` sequence lengths."""
    longest = max(len(ids[i]) for i in idx)
    return min(max_len, -(-longest // PAD_MULTIPLE) * PAD_MULTIPLE)


def collate(tok, ids, idx, device, length: int | None = None):
    """Pad a batch to its longest passage, or to ``length`` (static shapes for XLA)."""
    feats = {"input_ids": [ids[i] for i in idx]}
    if length is None:
        enc = tok.pad(feats, return_tensors="pt")
    else:
        enc = tok.pad(feats, padding="max_length", max_length=length, return_tensors="pt")
    return enc["input_ids"].to(device), enc["attention_mask"].to(device)


def train_stage(
    model,
    tok,
    texts,
    y,
    *,
    epochs: float,
    lr_lora: float,
    lr_head: float,
    bs: int,
    max_len: int,
    n_classes: int,
    amp_dtype,
    seed: int = 0,
    max_steps: int | None = None,
    log_every: int = 50,
    log=print,
):
    """One training stage; class-weighted CE, AdamW, 10% warm-up then cosine decay."""
    device = model.head.weight.device
    rng = np.random.default_rng(seed)
    ids = tokenize(tok, texts, max_len)
    y = torch.tensor(np.array(y), device=device)
    counts = torch.bincount(y, minlength=n_classes).float()
    class_w = (counts.sum() / (n_classes * counts.clamp(min=1))).to(device)

    per_epoch = math.ceil(len(ids) / bs)
    total = math.ceil(epochs * per_epoch)
    if max_steps is not None:
        total = min(total, max_steps)
    lora = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
    rest = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n]
    opt = torch.optim.AdamW(
        [{"params": lora, "lr": lr_lora}, {"params": rest, "lr": lr_head}], weight_decay=0.01
    )
    warm = max(1, int(0.1 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (
            (s + 1) / warm
            if s < warm
            else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
        ),
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

    model.train()
    step, skipped, run_loss, run_n, t0 = 0, 0, 0.0, 0, time.time()
    while step < total:
        for idx in batches(ids, bs, shuffle=True, rng=rng):
            if step >= total:
                break
            input_ids, mask = collate(tok, ids, idx, device)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(input_ids, mask)
            loss = nn.functional.cross_entropy(logits.float(), y[idx], weight=class_w)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1  # fp16 overflow in the backbone; drop this batch
            else:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(lora + rest, 1.0)
                scaler.step(opt)
                scaler.update()
                run_loss += loss.item()
                run_n += 1
            sched.step()
            step += 1
            if step % log_every == 0 or step == total:
                el = time.time() - t0
                log(
                    f"step {step}/{total} loss {run_loss / max(1, run_n):.4f} "
                    f"{el / step:.2f}s/step, ~{el / step * (total - step) / 60:.0f} min left, "
                    f"skipped {skipped}"
                )
                run_loss, run_n = 0.0, 0
    model.eval()
    return {"steps": total, "skipped": skipped, "seconds": time.time() - t0}


@torch.no_grad()
def predict_logits(
    model, tok, texts, *, bs: int, max_len: int, amp_dtype, static: bool = False
) -> np.ndarray:
    """Logits for ``texts``. ``static=True`` (XLA) pads every batch to ``bs`` rows and a length
    from ``static_len``; the padding rows are dropped again."""
    device = model.head.weight.device
    ids = tokenize(tok, texts, max_len)
    out = np.zeros((len(ids), model.head.out_features), np.float32)
    model.eval()
    for idx in batches(ids, bs, shuffle=False):
        rows = idx
        if static and len(idx) < bs:
            rows = np.concatenate([idx, np.repeat(idx[-1:], bs - len(idx))])
        length = static_len(ids, rows, max_len) if static else None
        input_ids, mask = collate(tok, ids, rows, device, length)
        with torch.autocast(
            device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"
        ):
            out[idx] = model(input_ids, mask).float().cpu().numpy()[: len(idx)]
    return out


def load_tokenizer(base_dir: str):
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def save_checkpoint(model, out_dir: str, meta: dict):
    from safetensors.torch import save_file

    os.makedirs(out_dir, exist_ok=True)
    save_file(
        {k: v.contiguous() for k, v in model.trainable_state().items()},
        os.path.join(out_dir, "lora_classifier.safetensors"),
    )
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)


def load_checkpoint(model, ckpt_dir: str):
    from safetensors.torch import load_file

    state = load_file(os.path.join(ckpt_dir, "lora_classifier.safetensors"))
    missing = model.load_state_dict(state, strict=False)
    assert not missing.unexpected_keys, missing.unexpected_keys
    return model


## LoRA logits for the test set, one member at a time

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    amp = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
else:
    device, amp = torch.device('cpu'), torch.float32
def log_softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=1, keepdims=True))


lora_logits = {}  # per member: the mean log-probabilities of its adapters
for name, lora_dir in LORA_DIRS.items():
    meta = json.load(open(f'{lora_dir}/meta.json'))
    assert meta['labels'] == list(CLASSES), f'{name}: label order differs from the SVM'
    cfg = meta['cfg']
    base = find_base(cfg.get('base', DEFAULT_BASE)) if ON_KAGGLE else os.environ.get('NTL_LORA_BASE', '../data/interim/tiny_qwen3')
    t0 = time.time()
    tok = load_tokenizer(base)
    # a trunk too large for one GPU (Qwen3-14B) is split over both
    device2 = torch.device('cuda:1') if torch.cuda.device_count() > 1 else None
    trunk = load_trunk(base, cfg['keep'], amp, device, device2=device2)
    lora_model = LoRAClassifier(trunk, cfg['layers'], len(CLASSES), cfg['r'], cfg['alpha'], 0.0, ckpt=False)
    logps = []
    for adapter in LORA_ADAPTERS[name]:
        assert json.load(open(f'{adapter}/meta.json'))['labels'] == list(CLASSES)
        load_checkpoint(lora_model, adapter)
        z = predict_logits(lora_model, tok, test.text, bs=16, max_len=cfg['max_len'], amp_dtype=amp)
        assert np.isfinite(z).all(), f'{adapter}: non-finite LoRA logits'
        logps.append(log_softmax(z))
    lora_logits[name] = np.mean(logps, axis=0)
    print(f'{name}: {len(logps)} adapter(s) in {time.time() - t0:.0f}s on {device} from {base}; '
          f"held-out balanced accuracy at training: fold 0 {meta.get('holdout_balanced_accuracy', float('nan')):.4f}, "
          f"all folds {meta.get('oof_balanced_accuracy', float('nan')):.4f}")
    del lora_model, trunk  # free the GPU for the next member
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

## Blend, balance, write `submission.parquet`

In [ ]:
def log_softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=1, keepdims=True))


logp = log_softmax(svm_scores * T_SVM)
for name, w_lora in LORA_WEIGHTS.items():
    logp = logp + w_lora * log_softmax(lora_logits[name])


def write_submission(logp):
    pred = CLASSES[sinkhorn(softmax(logp, 1.0), BALANCE_STRENGTH).argmax(axis=1)]
    submission = pd.DataFrame({'id': test.id, 'model': pred})
    assert submission.id.is_unique and len(submission) == len(test)
    assert submission.model.notna().all() and submission.model.isin(LABELS).all()
    submission.to_parquet(OUT, index=False)
    return submission


# the blend without the GLM member, written first: it stays the submission if the GLM stage fails
submission = write_submission(logp)
print(submission.model.value_counts().to_string())


## GLM-5.3 likelihood member

Scores the test passages the blend most suspects of being GLM (a share `GLM_COVER` of the test set, fewer if time is short) under the real GLM-5.3 and GLM-5.3-Flash, and adds a logistic regression on their likelihood features, fitted on train.csv's, to the blend. The scoring pipeline (`ntl-segrun`, `run_pipeline.py`, `features.py`, `segio.py`) is José Padilla's, from *0.689 | Real GLM-5.3 Likelihood on 2xT4*; the segment GGUFs are jamesliebel's mirrors of jinghengwu's quantization of Z.ai's open weights. The member's code is copied from `name_that_llm/glm_member.py`.

The models read each passage after the user-turn prefix of José Padilla's *0.716 | Robust GLM-5.3 Likelihood + User Prefix* (prvsiyan's idea): a request for one paragraph of about as many words as the passage has. The train features are the outputs of his `ntl-glm-prefix-ab` notebook, scored with the same prefix.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

GLM_COVER, GLM_WEIGHT, GLM_C = 0.2, 2.0, 0.05
GLM_PREFIX = '[gMASK]<sop><|system|>Reasoning Effort: Low<|user|>{user}<|assistant|><think></think>'
GLM_MAX_TEXT_TOKENS = 160
GLM_SEGMENTS = {"jamesliebel/glm53-iq3xxs-segs-a-mirror": [["glm53_iq3xxs-seg00-of-19-L0-6.gguf", 12753355264], ["glm53_iq3xxs-seg01-of-19-L6-10.gguf", 15651714688], ["glm53_iq3xxs-seg02-of-19-L10-14.gguf", 14583346912], ["glm53_iq3xxs-seg03-of-19-L14-18.gguf", 14583346912], ["glm53_iq3xxs-seg04-of-19-L18-22.gguf", 14583346912], ["glm53_iq3xxs-seg05-of-19-L22-26.gguf", 14583346912], ["glm53_iq3xxs-seg06-of-19-L26-30.gguf", 14583346912], ["glm53_iq3xxs-seg07-of-19-L30-34.gguf", 14583346912], ["glm53_iq3xxs-seg08-of-19-L34-38.gguf", 14583346912], ["glm53_iq3xxs-seg09-of-19-L38-42.gguf", 14583346912]], "jamesliebel/glm53-iq3xxs-segs-b-mirror": [["glm53_iq3xxs-seg10-of-19-L42-46.gguf", 14583346912], ["glm53_iq3xxs-seg11-of-19-L46-50.gguf", 14583346912], ["glm53_iq3xxs-seg12-of-19-L50-54.gguf", 14583346912], ["glm53_iq3xxs-seg13-of-19-L54-58.gguf", 14583346912], ["glm53_iq3xxs-seg14-of-19-L58-62.gguf", 14583346912], ["glm53_iq3xxs-seg15-of-19-L62-66.gguf", 14583346912], ["glm53_iq3xxs-seg16-of-19-L66-70.gguf", 14583346912], ["glm53_iq3xxs-seg17-of-19-L70-74.gguf", 14583346912], ["glm53_iq3xxs-seg18-of-19-L74-78.gguf", 16345433920]], "jamesliebel/glm53-flash-iq3xxs-segs-mirror": [["flash_iq3xxs-seg00-of-06-L0-8.gguf", 14879704352], ["flash_iq3xxs-seg01-of-06-L8-16.gguf", 22864262976], ["flash_iq3xxs-seg02-of-06-L16-24.gguf", 21821175680], ["flash_iq3xxs-seg03-of-06-L24-32.gguf", 21821175680], ["flash_iq3xxs-seg04-of-06-L32-40.gguf", 21821175680], ["flash_iq3xxs-seg05-of-06-L40-45.gguf", 14407253632]]}
GLM_TRAIN_FEATURES = 'ntl-glm-userturn-train-features'
GLM_FIXED_S, GLM_PER_PASSAGE_S, GLM_SAFETY = 1330, 4.75, 1.25
DEADLINE_S = 8.5 * 3600  # after T_START: Kaggle stops a submission at 9 h
GLM_CHECK_ROWS = 8  # interactive runs: train passages scored again, against their stored features
RERUN = bool(os.environ.get('KAGGLE_IS_COMPETITION_RERUN'))
GLM_WORK = Path('/kaggle/temp/glm_ll')  # pipeline code, passage list, state between segments
GLM_BIN = Path('/kaggle/temp/glm_ll_bin')
GLM_OUT = Path('/kaggle/working/glm_ll')  # features and run logs of the passages scored
for d in (GLM_WORK, GLM_BIN, GLM_OUT):
    d.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

GLM_TAGS = ("flash", "glm53")
GLM_LABELS = ("glm-5.3", "glm-5.3-flash")
DIFF_FEATURES = ("lp_mean", "lp_q50", "ent_mean", "top1", "logrank_mean", "fdg")


def likelihood_frame(flash: pd.DataFrame, glm53: pd.DataFrame, ids) -> pd.DataFrame:
    """Member inputs in ``ids`` order: each model's features (frames indexed by passage id as
    str), then the Flash - GLM-5.3 differences. A passage missing from either frame has NaNs."""
    blocks = [f.add_prefix(f"{tag}:") for tag, f in zip(GLM_TAGS, (flash, glm53))]
    frame = pd.concat(blocks, axis=1).reindex([str(i) for i in ids])
    for c in DIFF_FEATURES:
        frame[f"d:{c}"] = frame[f"flash:{c}"] - frame[f"glm53:{c}"]
    return frame


def fit_member(x: np.ndarray, y: np.ndarray, C: float):
    """Standardised features, then a multinomial logistic regression."""
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
    return model.fit(x, y)


def glm_mass(logp: np.ndarray, labels) -> np.ndarray:
    """The blend's probability of the two GLM labels, per passage (rows of log-probabilities)."""
    cols = [list(labels).index(c) for c in GLM_LABELS]
    z = np.exp(logp - logp.max(axis=1, keepdims=True))
    return (z[:, cols].sum(axis=1)) / z.sum(axis=1)


In [ ]:
%%writefile /kaggle/temp/glm_ll/segio.py
# Third-party code, copied unchanged from a public Kaggle notebook:
#   Jose Padilla (estopadilla), "0.689 | Real GLM-5.3 Likelihood on 2xT4"
#   https://www.kaggle.com/code/estopadilla/0-689-real-glm-5-3-likelihood-on-2xt4
# as carried in lightningv08's "0.702 | Soft Typography + GLM + Qwen3.8-27B LoRA".
# It drives the author's `ntl-segrun` binary (a llama.cpp patch) over layer-segment GGUFs.
# ruff: noqa
"""Read and write the binary files of ntl-segrun (see the header of tools/ntl-segrun/ntl-segrun.cpp)."""
import struct

import numpy as np


def write_texts(path, texts):
    with open(path, 'wb') as f:
        f.write(b'NTLT')
        f.write(struct.pack('<I', len(texts)))
        for t in texts:
            b = t.encode('utf-8')
            f.write(struct.pack('<I', len(b)))
            f.write(b)


def write_tokens(path, seqs):
    """seqs: list of (ids, n_prefix)."""
    with open(path, 'wb') as f:
        f.write(b'NTLK')
        f.write(struct.pack('<I', len(seqs)))
        for ids, n_prefix in seqs:
            ids = np.asarray(ids, dtype='<i4')
            f.write(struct.pack('<II', len(ids), n_prefix))
            f.write(ids.tobytes())


def read_tokens(path):
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLK'
        (n,) = struct.unpack('<I', f.read(4))
        for _ in range(n):
            nt, npre = struct.unpack('<II', f.read(8))
            ids = np.frombuffer(f.read(4 * nt), dtype='<i4')
            out.append((ids, npre))
    return out


def read_scores(path):
    """List of float32 arrays [n_scored, 4]: logprob, entropy, E[logp^2], rank."""
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLP'
        n, nf = struct.unpack('<II', f.read(8))
        for _ in range(n):
            (ns,) = struct.unpack('<I', f.read(4))
            out.append(np.frombuffer(f.read(4 * ns * nf), dtype='<f4').reshape(ns, nf))
    return out


def read_state(path):
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLS'
        dt, w, nt, nr = struct.unpack('<IIIQ', f.read(20))
        dtype = {0: '<f4', 1: '<f2'}.get(dt)
        if dtype is None:  # bf16
            raw = np.frombuffer(f.read(2 * w * nr), dtype='<u2').astype('<u4') << 16
            return raw.view('<f4').reshape(nr, w)
        return np.frombuffer(f.read(np.dtype(dtype).itemsize * w * nr), dtype=dtype).reshape(nr, w).astype('f4')


In [ ]:
%%writefile /kaggle/temp/glm_ll/features.py
# Third-party code, copied unchanged from a public Kaggle notebook:
#   Jose Padilla (estopadilla), "0.689 | Real GLM-5.3 Likelihood on 2xT4"
#   https://www.kaggle.com/code/estopadilla/0-689-real-glm-5-3-likelihood-on-2xt4
# as carried in lightningv08's "0.702 | Soft Typography + GLM + Qwen3.8-27B LoRA".
# It drives the author's `ntl-segrun` binary (a llama.cpp patch) over layer-segment GGUFs.
# ruff: noqa
"""Per-text likelihood features from the per-token scores of ntl-segrun.

Per scored token: lp = log-prob of the true token, H = entropy of the next-token distribution,
m2 = E_p[logp^2] (so Var_p[logp] = m2 - H^2), rank = how many tokens beat the true one (0 = top-1).
"""
import numpy as np
import pandas as pd


def one(s, n_chars):
    lp, h, m2, rank = s[:, 0].astype(np.float64), s[:, 1].astype(np.float64), s[:, 2].astype(np.float64), s[:, 3]
    n = len(lp)
    var = np.maximum(m2 - h * h, 1e-9)
    half = max(n // 2, 1)
    q = np.quantile(lp, [0.05, 0.10, 0.25, 0.50, 0.75, 0.90]) if n else np.zeros(6)
    return {
        'n_tok': n,
        'lp_mean': lp.mean(),
        'lp_std': lp.std(),
        'lp_q05': q[0], 'lp_q10': q[1], 'lp_q25': q[2], 'lp_q50': q[3], 'lp_q75': q[4], 'lp_q90': q[5],
        'lp_min': lp.min(),
        'ent_mean': h.mean(),
        'ent_std': h.std(),
        # Fast-DetectGPT analytic curvature: how far the text's log-likelihood sits above the model's own expectation
        'fdg': (lp + h).sum() / np.sqrt(var.sum()),
        'surprise_mean': (-lp - h).mean(),          # observed surprisal minus expected surprisal
        'z_mean': ((lp + h) / np.sqrt(var)).mean(),
        'logrank_mean': np.log1p(rank).mean(),
        'top1': (rank == 0).mean(),
        'top5': (rank < 5).mean(),
        'top20': (rank < 20).mean(),
        'rank_ge100': (rank >= 100).mean(),
        'lp_mean_first': lp[:half].mean(),
        'lp_mean_second': lp[half:].mean() if n > half else lp.mean(),
        'bpc': -lp.sum() / max(n_chars, 1) / np.log(2),
        'tok_per_char': n / max(n_chars, 1),
    }


def text_features(scores, texts):
    return pd.DataFrame([one(s, len(t)) for s, t in zip(scores, texts)])


In [ ]:
%%writefile /kaggle/temp/glm_ll/run_pipeline.py
# Third-party code, copied unchanged from a public Kaggle notebook:
#   Jose Padilla (estopadilla), "0.689 | Real GLM-5.3 Likelihood on 2xT4"
#   https://www.kaggle.com/code/estopadilla/0-689-real-glm-5-3-likelihood-on-2xt4
# as carried in lightningv08's "0.702 | Soft Typography + GLM + Qwen3.8-27B LoRA".
# The per-text user-turn prefix (--user-template, user_prefix_tokens, _user_prefix_pieces) is copied
# unchanged from the same author's run_pipeline.py in "0.716 | Robust GLM-5.3 Likelihood + User Prefix"
#   https://www.kaggle.com/code/estopadilla/0-716-robust-glm-5-3-likelihood-user-prefix
# It drives the author's `ntl-segrun` binary (a llama.cpp patch) over layer-segment GGUFs.
# ruff: noqa
"""Score text likelihood by streaming each model segment once per state chunk."""

import argparse
import glob
import json
import os
import re
import shutil
import struct
import subprocess
import sys
import threading
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import features  # noqa: E402
import segio  # noqa: E402

CHAT_PREFIX = '[gMASK]<sop><|user|><|assistant|><think></think>'


def seg_index(path):
    m = re.search(r'seg(\d+)-of-(\d+)', os.path.basename(path))
    assert m, path
    return int(m.group(1)), int(m.group(2))


def gguf_kv_u32(path, key):
    """Read one uint32 metadata value from a GGUF header without the gguf package."""
    with open(path, 'rb') as f:
        assert f.read(4) == b'GGUF'
        _ver, _nt, nkv = struct.unpack('<IQQ', f.read(20))

        def rstr():
            (n,) = struct.unpack('<Q', f.read(8))
            return f.read(n).decode('utf-8', 'replace')

        sizes = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}

        def skip(t):
            if t == 8:
                rstr()
            elif t == 9:
                it, n = struct.unpack('<IQ', f.read(12))
                if it in sizes:
                    f.seek(sizes[it] * n, 1)
                else:
                    for _ in range(n):
                        skip(it)
            else:
                f.seek(sizes[t], 1)

        for _ in range(nkv):
            k = rstr()
            (t,) = struct.unpack('<I', f.read(4))
            if k == key:
                assert t == 4, (key, t)
                return struct.unpack('<I', f.read(4))[0]
            skip(t)
    return None


class EarlyFailure(RuntimeError):
    """ntl-segrun failed before writing any output (model load or context reservation, e.g. out of GPU memory)."""


_PREFETCHED = {}


def prefetch(path):
    """Read a file once in a daemon thread (page cache warm-up for the next mmap); each file at most once."""
    path = os.path.realpath(path)
    if path in _PREFETCHED:
        return
    info = _PREFETCHED[path] = {'s': None}

    def read():
        t0 = time.time()
        with open(path, 'rb', buffering=0) as f:
            try:
                os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_SEQUENTIAL)
            except (AttributeError, OSError):
                pass
            buf = bytearray(16 << 20)
            while f.readinto(buf):
                pass
        info['s'] = time.time() - t0
    threading.Thread(target=read, daemon=True).start()


def run(cmd, log, on_ready=None):
    """Run one ntl-segrun call; on_ready() is called when it prints "context ready" (the model is loaded)."""
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    lines = []
    for line in p.stdout:
        lines.append(line)
        if on_ready is not None and 'context ready' in line:
            on_ready()
            on_ready = None
    p.wait()
    out = ''.join(lines)
    with open(log, 'a') as lf:
        lf.write('$ ' + ' '.join(cmd) + '\n' + out)
    dt = time.time() - t0
    if p.returncode != 0:
        print('\n'.join(out.splitlines()[-40:]), flush=True)
        err = EarlyFailure if 'context ready' not in out else RuntimeError
        raise err(f'failed ({p.returncode}): {" ".join(cmd)}')
    return dt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--bin', required=True)
    ap.add_argument('--segs', required=True, help='JSON list of the pinned segment paths')
    ap.add_argument('--csv', required=True)
    ap.add_argument('--text-col', default='text')
    ap.add_argument('--id-col', default='id')
    ap.add_argument('--out', required=True)
    ap.add_argument('--tag', required=True)
    ap.add_argument('--work', default='/tmp/ntl')
    ap.add_argument('--prefix', default=CHAT_PREFIX)
    ap.add_argument('--max-text-tokens', type=int, default=320)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--state-dtype', default='auto', help='auto | f32 | f16 | bf16')
    ap.add_argument('--threads', type=int, default=min(4, os.cpu_count() or 4))
    ap.add_argument('--limit', type=int, default=-1)
    ap.add_argument('--disk-frac', type=float, default=0.8, help='share of the free work disk the state may use')
    ap.add_argument('--pack', action='store_true', help='sort by length and pack padded decodes (ntl-segrun --pack)')
    ap.add_argument('--ubatch-div', type=int, default=1, help='n_ubatch = n_batch / P (pipeline parallel on 2 GPUs)')
    ap.add_argument('--prefetch', action='store_true', help='read the next segment while the current one computes')
    ap.add_argument('--prefetch-after', default='', help='with --prefetch: file to read during the last segment')
    ap.add_argument('--user-template', default='Write a single paragraph of about {n_words} words, with no title or headings.',
                    help="with '{user}' in --prefix: the user turn of each text ({n_words} = its word count)")
    a = ap.parse_args()
    main_run(a)


def user_prefix_tokens(a, segrun, tok_model, texts, tx, tk, log):
    """A prefix with a user turn that depends on the text: --prefix holds '{user}', replaced for each text by
    --user-template formatted with n_words (the text's word count), e.g. prvsiyan's GLM prefix
    '[gMASK]<sop><|system|>Reasoning Effort: Low<|user|>{user}<|assistant|><think></think>'.
    ntl-segrun tokenize takes one prefix for all texts, so the tokens are put together from two calls (exact: the
    special tokens split the string before BPE, so each plain-text piece is tokenized on its own):
      full = tokenize(texts, prefix with an empty user turn) = H + T + text   (H = up to '{user}', T = the rest)
      user = tokenize(user turns, prefix H)                    = H + U_i
    and each text gets H + U_i + T + text, with n_prefix = len(H + U_i + T). Checked on the first 3 texts
    against a tokenization with their whole prefix; if that check fails (a tokenizer where the pieces are not
    independent), the exact fallback tokenizes each group of texts that share the user turn with its whole prefix."""
    users = [a.user_template.format(n_words=len(t.split())) for t in texts]
    try:
        return _user_prefix_pieces(a, segrun, tok_model, texts, users, tx, tk, log)
    except AssertionError as e:
        # exact fallback: one tokenize call per distinct user turn (texts with the same word count share it)
        print(f'[{a.tag}] user-turn prefix: piecewise tokens rejected ({e}); tokenizing by groups', flush=True)
    groups = {}
    for i, u in enumerate(users):
        groups.setdefault(u, []).append(i)
    seqs = [None] * len(texts)
    for j, (u, idx) in enumerate(groups.items()):
        gx, gk = tk + f'.g{j}.bin', tk + f'.g{j}_tok.bin'
        segio.write_texts(gx, [texts[i] for i in idx])
        run([segrun, 'tokenize', '--model', tok_model, '--texts', gx, '--out', gk, '--prefix', a.prefix.replace('{user}', u),
             '--max-text-tokens', str(a.max_text_tokens)], log)
        for i, sq in zip(idx, segio.read_tokens(gk)):
            seqs[i] = sq
        os.remove(gx); os.remove(gk)
    print(f'[{a.tag}] user-turn prefix: {len(groups)} groups, prefix {min(s[1] for s in seqs)}-{max(s[1] for s in seqs)} '
          f'tokens; e.g. {users[0]!r}', flush=True)
    return seqs


def _user_prefix_pieces(a, segrun, tok_model, texts, users, tx, tk, log):
    head = a.prefix.split('{user}')[0]
    run([segrun, 'tokenize', '--model', tok_model, '--texts', tx, '--out', tk, '--prefix', a.prefix.replace('{user}', ''),
         '--max-text-tokens', str(a.max_text_tokens)], log)
    full = segio.read_tokens(tk)
    ux, uk = tk + '.user_texts.bin', tk + '.user_tok.bin'
    segio.write_texts(ux, users)
    run([segrun, 'tokenize', '--model', tok_model, '--texts', ux, '--out', uk, '--prefix', head,
         '--max-text-tokens', '100000'], log)
    user = segio.read_tokens(uk)
    n_h = user[0][1]
    seqs = []
    for (f_ids, f_pre), (u_ids, u_pre) in zip(full, user):
        assert u_pre == n_h and f_pre >= n_h and list(f_ids[:n_h]) == list(u_ids[:n_h]), 'prefix head differs'
        seqs.append((np.concatenate([u_ids, f_ids[n_h:]]).astype('<i4'), len(u_ids) + f_pre - n_h))
    for i in range(min(3, len(texts))):
        one_x, one_k = tk + f'.check{i}.bin', tk + f'.check{i}_tok.bin'
        segio.write_texts(one_x, [texts[i]])
        run([segrun, 'tokenize', '--model', tok_model, '--texts', one_x, '--out', one_k,
             '--prefix', a.prefix.replace('{user}', users[i]), '--max-text-tokens', str(a.max_text_tokens)], log)
        ids, pre = segio.read_tokens(one_k)[0]
        assert pre == seqs[i][1] and list(ids) == list(seqs[i][0]), f'user-prefix tokens differ for text {i}'
    print(f'[{a.tag}] user-turn prefix: head {n_h} tokens, prefix {min(s[1] for s in seqs)}-{max(s[1] for s in seqs)} '
          f'tokens (pieces checked on {min(3, len(texts))} texts); e.g. {users[0]!r}', flush=True)
    return seqs


def main_run(a):

    t_start = time.time()

    os.makedirs(a.out, exist_ok=True)
    os.makedirs(a.work, exist_ok=True)
    log = os.path.join(a.out, f'{a.tag}_run.log')
    segrun = os.path.join(a.bin, 'ntl-segrun')

    segs = sorted(json.loads(a.segs), key=lambda p: seg_index(p)[0])
    assert segs, a.segs
    n_seg = seg_index(segs[0])[1]
    assert [seg_index(p)[0] for p in segs] == list(range(n_seg)), segs
    ranges = [(gguf_kv_u32(p, 'ntl.layer_start'), gguf_kv_u32(p, 'ntl.layer_end')) for p in segs]
    print(f'[{a.tag}] {n_seg} segments, layers {ranges}', flush=True)
    for (s0, e0), (s1, _e1) in zip(ranges, ranges[1:]):
        assert e0 == s1, ranges

    df = pd.read_csv(a.csv)
    if a.limit > 0:
        df = df.iloc[:a.limit]
    texts = df[a.text_col].astype(str).tolist()
    tx = os.path.join(a.work, f'{a.tag}_texts.bin')
    tk = os.path.join(a.work, f'{a.tag}_tok.bin')
    segio.write_texts(tx, texts)
    if '{user}' in a.prefix:
        seqs = user_prefix_tokens(a, segrun, segs[0], texts, tx, tk, log)
    else:
        run([segrun, 'tokenize', '--model', segs[0], '--texts', tx, '--out', tk, '--prefix', a.prefix,
             '--max-text-tokens', str(a.max_text_tokens)], log)
        seqs = segio.read_tokens(tk)
    n_rows = np.array([len(ids) for ids, _ in seqs])
    # processing order: longest first when packing (similar lengths share a decode, little padding)
    order = np.argsort(-n_rows, kind='stable') if a.pack else np.arange(len(seqs))
    print(f'[{a.tag}] {len(seqs)} texts, {n_rows.sum()} rows, max {n_rows.max()}, prefix {seqs[0][1]}', flush=True)

    # state width: glm5-next hc*n_embd, glm-dsa n_embd (read from the first segment's metadata)
    n_embd = gguf_kv_u32(segs[0], 'glm5-next.embedding_length')
    if n_embd is not None:
        width = gguf_kv_u32(segs[0], 'glm5-next.hyper_connection.count') * n_embd
    else:
        width = gguf_kv_u32(segs[0], 'glm-dsa.embedding_length')
    free = shutil.disk_usage(a.work).free
    if a.state_dtype == 'auto':
        # exact f32 if the whole state fits in one chunk, else bf16 (half the disk, no overflow risk)
        a.state_dtype = 'f32' if n_rows.sum() * width * 4 <= free * a.disk_frac else 'bf16'
        print(f'[{a.tag}] state dtype auto -> {a.state_dtype}', flush=True)
    el = 2 if a.state_dtype in ('f16', 'bf16') else 4
    max_rows = int(free * a.disk_frac / (width * el))
    chunks, cur, cur_rows = [], [], 0
    for i in order:
        r = n_rows[i]
        if cur and cur_rows + r > max_rows:
            chunks.append(cur)
            cur, cur_rows = [], 0
        cur.append(i)
        cur_rows += r
    chunks.append(cur)
    print(f'[{a.tag}] state width {width}, {el} B/value, {free/2**30:.1f} GiB free on {a.work} -> '
          f'{len(chunks)} chunk(s)', flush=True)

    multi_gpu = False
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
        multi_gpu = out.count('GPU ') > 1
    except FileNotFoundError:
        pass

    timing = {'tag': a.tag, 'n_texts': len(seqs), 'n_rows': int(n_rows.sum()), 'chunks': [], 'segments': segs,
              'ranges': ranges, 'seq_batch': a.seq_batch, 'state_dtype': a.state_dtype, 'prefix': a.prefix,
              'pack': a.pack, 'ubatch_div': a.ubatch_div}
    all_scores = [None] * len(seqs)
    st = os.path.join(a.work, f'{a.tag}_state.bin')
    done = 0
    for ci, idx in enumerate(chunks):
        ctk = os.path.join(a.work, f'{a.tag}_tok_c{ci}.bin')
        segio.write_tokens(ctk, [seqs[i] for i in idx])
        sc = os.path.join(a.work, f'{a.tag}_scores_c{ci}.bin')
        seg_t = []
        for k, p in enumerate(segs):
            while True:
                cmd = [segrun, 'run', '--model', p, '--tokens', ctk, '--threads', str(a.threads),
                       '--seq-batch', str(a.seq_batch), '--state-dtype', a.state_dtype]
                if multi_gpu:
                    cmd.append('--split-mid')
                if a.pack:
                    cmd.append('--pack')
                if a.ubatch_div > 1:
                    cmd += ['--ubatch-div', str(a.ubatch_div)]
                if k > 0:
                    cmd += ['--state-in', st]
                cmd += ['--out', sc if k == n_seg - 1 else st]
                nxt = (segs[k + 1] if k + 1 < n_seg else segs[0] if ci + 1 < len(chunks) else a.prefetch_after)
                ready = (lambda f=nxt: prefetch(f)) if a.prefetch and nxt else None
                try:
                    dt = run(cmd, log, on_ready=ready)
                    break
                except EarlyFailure:
                    # nothing written yet (the input state is intact): retry with half the texts per decode
                    if a.seq_batch == 1:
                        raise
                    a.seq_batch //= 2
                    timing['seq_batch_reduced_to'] = a.seq_batch
                    print(f'[{a.tag}] segment {k+1} failed before writing: retrying with seq-batch {a.seq_batch}', flush=True)
            seg_t.append(dt)
            print(f'[{a.tag}] chunk {ci+1}/{len(chunks)} segment {k+1}/{n_seg} {os.path.basename(p)}: {dt:.0f} s '
                  f'(total {time.time()-t_start:.0f} s)', flush=True)
        for i, s in zip(idx, segio.read_scores(sc)):
            all_scores[i] = s
        done += len(idx)
        timing['chunks'].append({'n_texts': len(idx), 'seg_seconds': seg_t})
        for f in (ctk, sc, st):
            if os.path.exists(f):
                os.remove(f)

    ok = [i for i, s in enumerate(all_scores) if s is not None]
    off = np.zeros(len(ok) + 1, dtype=np.int64)
    off[1:] = np.cumsum([len(all_scores[i]) for i in ok])
    np.savez_compressed(os.path.join(a.out, f'{a.tag}_scores.npz'),
                        ids=df[a.id_col].astype(str).values[ok],
                        scores=np.concatenate([all_scores[i] for i in ok]).astype(np.float32),
                        offsets=off,
                        n_prefix=np.array([seqs[i][1] for i in ok]),
                        tokens=np.concatenate([seqs[i][0][seqs[i][1]:] for i in ok]).astype(np.int32))
    feats = features.text_features([all_scores[i] for i in ok], [texts[i] for i in ok])
    feats.insert(0, a.id_col, df[a.id_col].astype(str).values[ok])
    feats.to_csv(os.path.join(a.out, f'{a.tag}_features.csv'), index=False)
    timing['total_s'] = time.time() - t_start
    if a.prefetch:
        timing['prefetch_s'] = {os.path.basename(k): v['s'] for k, v in _PREFETCHED.items()}
    timing['n_done'] = len(ok)
    with open(os.path.join(a.out, f'{a.tag}_timing.json'), 'w') as f:
        json.dump(timing, f, indent=1)
    print(f'[{a.tag}] done: {len(ok)}/{len(seqs)} texts in {timing["total_s"]:.0f} s', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
def glm_inputs():
    # segments (exact sizes), the ntl-segrun binary, and train.csv's stored features
    flash, glm53 = [], []
    for ref, files in GLM_SEGMENTS.items():
        for name, size in files:
            hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
            assert hits, f'{name} missing: attach {ref}'
            assert os.path.getsize(hits[0]) == size, f'{hits[0]}: not {size} bytes'
            (flash if name.startswith('flash_') else glm53).append(hits[0])
    bins = sorted(glob.glob('/kaggle/input/**/ntl-bin/ntl-segrun', recursive=True))
    assert bins, 'attach the output of estopadilla/ntl-glm-likelihood-train (ntl-bin/)'
    stored = {}
    for tag in GLM_TAGS:
        hits = sorted(glob.glob(f'/kaggle/input/**/{GLM_TRAIN_FEATURES}/**/{tag}_features.csv',
                                recursive=True))
        assert hits, f'attach the output of {GLM_TRAIN_FEATURES} ({tag}_features.csv)'
        stored[tag] = pd.read_csv(hits[0], dtype={'id': str}).set_index('id')
    return sorted(flash), sorted(glm53), Path(bins[0]).parent, stored


def run_glm(csv, flash_segs, glm53_segs, bin_dir):
    # both models over the passages in csv; each run stops at the deadline
    shutil.copyfile(bin_dir / 'ntl-segrun', GLM_BIN / 'ntl-segrun')
    (GLM_BIN / 'ntl-segrun').chmod(0o755)
    # the binary was built against CUDA 13; its runtime ships next to it
    env = dict(os.environ, LD_LIBRARY_PATH=f"{bin_dir / 'lib'}:{os.environ.get('LD_LIBRARY_PATH', '')}")
    for tag, segs in (('flash', flash_segs), ('glm53', glm53_segs)):
        cmd = [sys.executable, str(GLM_WORK / 'run_pipeline.py'), '--bin', str(GLM_BIN),
               '--segs', json.dumps(segs), '--csv', str(csv), '--out', str(GLM_OUT), '--tag', tag,
               '--work', str(GLM_WORK / 'state'), '--seq-batch', '8', '--state-dtype', 'f32',
               '--max-text-tokens', str(GLM_MAX_TEXT_TOKENS), '--prefix', GLM_PREFIX]
        t = time.time()
        # its own process group, so a timeout also stops ntl-segrun under it
        p = subprocess.Popen(cmd, env=env, start_new_session=True)
        try:
            code = p.wait(timeout=max(60.0, DEADLINE_S - (time.time() - T_START)))
        except BaseException:
            os.killpg(p.pid, 9)
            p.wait()
            raise
        assert code == 0, f'{tag}: run_pipeline failed ({code}), see {GLM_OUT}/{tag}_run.log'
        print(f'{tag}: {time.time() - t:.0f} s', flush=True)
    return {tag: pd.read_csv(GLM_OUT / f'{tag}_features.csv', dtype={'id': str}).set_index('id')
            for tag in GLM_TAGS}


def glm_stage(logp):
    # logp plus the member's log-probabilities on the passages scored (None: no time to score any)
    t0 = time.time()
    flash_segs, glm53_segs, bin_dir, stored = glm_inputs()
    xtr = likelihood_frame(stored['flash'], stored['glm53'], train.id).to_numpy()
    ok = np.isfinite(xtr).all(axis=1)
    assert ok.mean() > 0.99, f'{(~ok).sum()} train passages have no stored likelihood features'
    member = fit_member(xtr[ok], train.model.to_numpy()[ok], GLM_C)
    assert list(member.classes_) == list(CLASSES)

    # the passages the blend most suspects of being GLM, as many as the cover and the time allow
    left = DEADLINE_S - (time.time() - T_START)
    n_cover = max(1, round(GLM_COVER * len(test)))
    n_time = int((left / GLM_SAFETY - GLM_FIXED_S) / GLM_PER_PASSAGE_S)
    n = min(n_cover, n_time)
    print(f'GLM stage: {left / 3600:.2f} h left; scoring {max(n, 0)} of {len(test)} test passages '
          f'(cover allows {n_cover}, time {n_time})', flush=True)
    if n < 1:
        return None
    sel = np.argsort(-glm_mass(logp, CLASSES), kind='stable')[:n]
    # ids by position, so that no id is read back as a number
    texts = pd.DataFrame({'id': [f'test:{i}' for i in range(n)], 'text': test.text.to_numpy()[sel]})
    k = 0 if RERUN else GLM_CHECK_ROWS
    check = pd.DataFrame({'id': [f'train:{i}' for i in range(k)], 'text': train.text.to_numpy()[:k]})
    csv = GLM_WORK / 'texts.csv'
    pd.concat([texts, check], ignore_index=True).to_csv(csv, index=False)
    if 'torch' in globals() and torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
    print(subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv'],
                         capture_output=True, text=True).stdout, flush=True)
    scored = run_glm(csv, flash_segs, glm53_segs, bin_dir)

    if len(check):
        # the same passages as in train.csv's run: same features, same member prediction
        a = likelihood_frame(scored['flash'], scored['glm53'], check.id)
        b = likelihood_frame(stored['flash'], stored['glm53'], train.id.iloc[:len(check)])
        gap = (a.to_numpy() - b.to_numpy())
        print('check passages, largest gap to the stored features:',
              {c: float(np.abs(gap[:, j]).max()) for j, c in enumerate(a.columns) if c.endswith('lp_mean')},
              f'all columns {np.nanmax(np.abs(gap)):.4f}; same member prediction for '
              f'{(member.predict(a.to_numpy()) == member.predict(b.to_numpy())).sum()} of {len(check)}', flush=True)
        assert np.isfinite(gap).all() and np.abs(gap[:, list(a.columns).index('flash:lp_mean')]).max() < 0.05

    x = likelihood_frame(scored['flash'], scored['glm53'], texts.id).to_numpy()
    ok = np.isfinite(x).all(axis=1)
    out = logp.copy()
    out[sel[ok]] += GLM_WEIGHT * member.predict_log_proba(x[ok])
    print(f'GLM stage: {ok.sum()} of {n} passages scored in {time.time() - t0:.0f} s', flush=True)
    return out


logp_glm = None
try:
    logp_glm = glm_stage(logp)
except Exception as e:
    if not RERUN:
        raise
    print(f'GLM stage failed; the submission stays the blend without it: {e!r}', flush=True)
if logp_glm is not None:
    before = submission.model.to_numpy()
    submission = write_submission(logp_glm)
    print(f'the GLM member changed {(submission.model.to_numpy() != before).sum()} predictions')
    print(submission.model.value_counts().to_string())
